### Module 0 (Performance demo [Python loop vs NumPy on 1 million elements])


In [13]:
# A Python loop summing 1 million random floats
import random
import time

start = time.perf_counter()
total = sum(random.uniform(1.0, 5.0) for _ in range(1_000_000))
end = time.perf_counter()
python_time = end - start

print("Sum: ", total)
print("Time Taken: ", python_time, "seconds")


Sum:  2999732.386264909
Time Taken:  0.6320683000376448 seconds


In [14]:
# NumPy summing 1 million random floats
import numpy as np

start_1 = time.perf_counter()
rng = np.random.default_rng()
arr = rng.uniform(1.0, 5.0, size=1_000_000)
total_1 = np.sum(arr)
end_1 = time.perf_counter()
numpy_time = end_1 - start_1

print("Sum: ", total_1)
print("Time Taken: ", numpy_time, "seconds")


Sum:  3000608.501030077
Time Taken:  0.12915540003450587 seconds


In [15]:
speedup_ratio = python_time / numpy_time
print(f"Speedup ratio (Python / NumPy): {speedup_ratio:.2f}x")


Speedup ratio (Python / NumPy): 4.89x


### Why NumPy is Faster (Hardware-Level Chain)

The timing gap is not just "NumPy is faster"; it comes from a chain of low-level properties:

1. **Fixed dtypes**: NumPy arrays store elements with one known type (`float64` here), so the CPU does not need per-element type checks.
2. **Contiguous memory**: values are packed in adjacent memory, enabling efficient sequential reads.
3. **Better cache behavior**: sequential access is cache-friendly, so fewer expensive memory fetches are needed.
4. **Vectorized native loops + SIMD**: NumPy executes compiled loops (C-level) and can apply SIMD instructions to process multiple numbers per CPU instruction.

In contrast, Python loops over Python objects with interpreter overhead per element, which makes large batch operations much slower.


### Module 0 Add-on: Fair Benchmark Using the Same Random Numbers

I put this section **below** my original Module 0 cells on purpose.

My thought process:
- I want to keep my first benchmark pass visible as part of my learning trail.
- This add-on is my refinement, not a replacement: I fixed benchmark fairness by making both methods sum the **same values**.
- Keeping it below preserves my narrative order: first attempt -> insight from review -> improved experiment.
- If I show this in an interview or portfolio review, it demonstrates how I think scientifically: I noticed a confounder (different inputs), then controlled it with a better setup.

So this lower section is my "v2 benchmark" under controlled conditions.


In [16]:
# Fair benchmark: both methods use the same random values
rng_fair = np.random.default_rng(42)
shared_arr = rng_fair.uniform(1.0, 5.0, size=1_000_000)
python_values = shared_arr.tolist()  # convert once so Python sums plain Python floats

# Python sum (same data)
start_py_fair = time.perf_counter()
python_total_fair = sum(python_values)
end_py_fair = time.perf_counter()
python_time_fair = end_py_fair - start_py_fair

# NumPy sum (same data)
start_np_fair = time.perf_counter()
numpy_total_fair = np.sum(shared_arr)
end_np_fair = time.perf_counter()
numpy_time_fair = end_np_fair - start_np_fair

speedup_ratio_fair = python_time_fair / numpy_time_fair

print(f"Python sum (same data): {python_total_fair:.6f}")
print(f"NumPy sum  (same data): {numpy_total_fair:.6f}")
print(f"Python time: {python_time_fair:.6f}s")
print(f"NumPy time:  {numpy_time_fair:.6f}s")
print(f"Fair speedup ratio (Python / NumPy): {speedup_ratio_fair:.2f}x")
print(f"Absolute sum difference: {abs(python_total_fair - numpy_total_fair):.12f}")


Python sum (same data): 3000105.904696
NumPy sum  (same data): 3000105.904696
Python time: 0.020420s
NumPy time:  0.002317s
Fair speedup ratio (Python / NumPy): 8.81x
Absolute sum difference: 0.000000000000


### Module 1 (Dataset setup and array inspection)


In [17]:
# Module 1: load the operational export and establish lightweight Python-side config
column_config = {
    "user_id": 0,
    "session_duration_sec": 1,
    "pages_viewed": 2,
    "purchase_amount": 3,
    "region_id": 4,
}
expected_shape = (500, 5)
raw_column_labels = [f"raw::{name}" for name in column_config]
known_region_ids = {0, 1, 2, 3, 4}

user_activity_struct = np.genfromtxt(
    "user_events.csv",
    delimiter=",",
    skip_header=1,
    dtype=[
        ("user_id", "i4"),
        ("session_duration_sec", "f8"),
        ("pages_viewed", "i4"),
        ("purchase_amount", "f8"),
        ("region_id", "i4"),
    ],
)

# A regular 2D ndarray can only have ONE dtype, so I cast to float64 for matrix math.
user_activity_data = np.column_stack(
    [
        user_activity_struct["user_id"].astype(np.float64),
        user_activity_struct["session_duration_sec"],
        user_activity_struct["pages_viewed"].astype(np.float64),
        user_activity_struct["purchase_amount"],
        user_activity_struct["region_id"].astype(np.float64),
    ]
)

observed_region_ids = set(user_activity_struct["region_id"].tolist())
assert user_activity_data.shape == expected_shape
assert observed_region_ids.issubset(known_region_ids)

print("Structured array dtype (preserves per-column schema):")
print(user_activity_struct.dtype)
print("2D matrix shape/dtype (for math):", user_activity_data.shape, user_activity_data.dtype)
print("Column config:", column_config)
print("Raw column labels:", raw_column_labels)
print("Observed region ids:", observed_region_ids)
print("Structured preview (first 3 rows):", user_activity_struct[:3])
print("2D preview (first 3 rows):\n", user_activity_data[:3])


Structured array dtype (preserves per-column schema):
[('user_id', '<i4'), ('session_duration_sec', '<f8'), ('pages_viewed', '<i4'), ('purchase_amount', '<f8'), ('region_id', '<i4')]
2D matrix shape/dtype (for math): (500, 5) float64
Column config: {'user_id': 0, 'session_duration_sec': 1, 'pages_viewed': 2, 'purchase_amount': 3, 'region_id': 4}
Raw column labels: ['raw::user_id', 'raw::session_duration_sec', 'raw::pages_viewed', 'raw::purchase_amount', 'raw::region_id']
Observed region ids: {0, 1, 2, 3, 4}
Structured preview (first 3 rows): [(1, 444.1, 5, 0., 4) (2, 207.3, 3, 0., 3) (3, 532.3, 7, 0., 1)]
2D preview (first 3 rows):
 [[  1.  444.1   5.    0.    4. ]
 [  2.  207.3   3.    0.    3. ]
 [  3.  532.3   7.    0.    1. ]]


In [18]:
# Printing all six array inspection properties, plus the nbytes formula check
computed_nbytes = user_activity_data.size * user_activity_data.itemsize

print("Dimension:", user_activity_data.ndim)
print("Shape:", user_activity_data.shape)
print("Dtype:", user_activity_data.dtype)
print("Item Size:", user_activity_data.itemsize)
print("Size:", user_activity_data.size)
print("No of bytes:", user_activity_data.nbytes)
print("Formula check (size * itemsize):", computed_nbytes)
print("nbytes formula verified:", computed_nbytes == user_activity_data.nbytes)


Dimension: 2
Shape: (500, 5)
Dtype: float64
Item Size: 8
Size: 2500
No of bytes: 20000
Formula check (size * itemsize): 20000
nbytes formula verified: True


In [19]:
# Reshaping the same dataset into 1D, 2D, and 3D views
user_activity_data_1d = user_activity_data.reshape(-1)
user_activity_data_2d = user_activity_data.reshape(100, 25)
user_activity_data_3d = user_activity_data.reshape(10, 10, 25)
three_d_convention = ("blocks", "rows per block", "columns per row")

print("1D reshaped view:", user_activity_data_1d.shape)
print("2D reshaped view:", user_activity_data_2d.shape)
print("3D reshaped view:", user_activity_data_3d.shape)
print("3D convention:", three_d_convention)
print("One 3D block shape:", user_activity_data_3d[0].shape)


1D reshaped view: (2500,)
2D reshaped view: (100, 25)
3D reshaped view: (10, 10, 25)
3D convention: ('blocks', 'rows per block', 'columns per row')
One 3D block shape: (10, 25)


In [20]:
# Module 1: explicit array creation (1D, 2D, 3D)
one_d_demo = np.array([1, 2, 3, 4], dtype=np.int32)
two_d_demo = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int32)
three_d_demo = np.array(
    [
        [[1, 2], [3, 4]],
        [[5, 6], [7, 8]],
    ],
    dtype=np.int32,
)

print("1D shape:", one_d_demo.shape)
print("2D shape:", two_d_demo.shape)
print("3D shape:", three_d_demo.shape)


1D shape: (4,)
2D shape: (2, 3)
3D shape: (2, 2, 2)


In [21]:
# Module 1: zeros, ones, full, arange, random, and empty_like
zeros_demo = np.zeros((3, 3), dtype=np.float64)
ones_demo = np.ones((2, 4), dtype=np.float64)
full_demo = np.full((2, 3), fill_value=-1, dtype=np.int32)
user_id_sequence = np.arange(1, 11, dtype=np.int32)
rand_demo = np.random.rand(2, 3)
randint_demo = np.random.randint(0, 10, size=(2, 3))
empty_like_demo = np.empty_like(user_activity_data)

print("zeros shape:", zeros_demo.shape)
print("ones shape:", ones_demo.shape)
print("full sample row:", full_demo[0])
print("arange ids:", user_id_sequence)
print("random float sample:\n", rand_demo)
print("random int sample:\n", randint_demo)
print("empty_like shape/dtype:", empty_like_demo.shape, empty_like_demo.dtype)


zeros shape: (3, 3)
ones shape: (2, 4)
full sample row: [-1 -1 -1]
arange ids: [ 1  2  3  4  5  6  7  8  9 10]
random float sample:
 [[0.17713189 0.23919842 0.41024582]
 [0.93005687 0.78766208 0.8271349 ]]
random int sample:
 [[0 4 7]
 [7 3 1]]
empty_like shape/dtype: (500, 5) float64


### Module 1 Add-on: `np.identity()` vs `np.eye()`

`np.identity(n)` only creates a square identity matrix.
`np.eye(rows, cols)` can create the square version too, but it also supports non-square matrices, which is why it is more flexible.


In [22]:
identity_demo = np.identity(4, dtype=np.float64)
eye_square_demo = np.eye(4, dtype=np.float64)
eye_rect_demo = np.eye(3, 5, dtype=np.float64)

print("identity shape:", identity_demo.shape)
print("eye (square) shape:", eye_square_demo.shape)
print("eye (rectangular) shape:", eye_rect_demo.shape)
print("identity equals eye square:", np.array_equal(identity_demo, eye_square_demo))


identity shape: (4, 4)
eye (square) shape: (4, 4)
eye (rectangular) shape: (3, 5)
identity equals eye square: True


**Module 1 production note**

In production, this `np.genfromtxt()` call would usually be replaced by a database export, a SQLAlchemy query, or a warehouse extract job. I keep both `user_activity_struct` and `user_activity_data` because they solve different needs: the structured array preserves per-column schema, while the 2D `float64` matrix is the version that makes vectorized math and matrix multiplication straightforward.


### Module 2 (ETL transformation and feature engineering)


In [23]:
# Module 2: inspect slices, select sample users, and create masks
# I call .copy() before mutation so feature engineering does not accidentally overwrite the raw export.
raw_feature_block = user_activity_data[:, 1:4].astype(np.float64).copy()
sample_indices = np.array([0, 24, 149, 320], dtype=np.int32)
sample_users = user_activity_data[sample_indices]
first_user_full_row = user_activity_data[0, :]
pages_column = user_activity_data[:, column_config["pages_viewed"]]
step_slice_demo = user_activity_data[0, 1:4:2]

high_value_mask = user_activity_data[:, column_config["purchase_amount"]] > 100.0
high_value_users = user_activity_data[high_value_mask]
positive_session_mask = user_activity_data[:, column_config["session_duration_sec"]] > 0

print("Sample indices:", sample_indices)
print("Sample user rows:\n", sample_users)
print("First user full row:", first_user_full_row)
print("Pages viewed column (first 5):", pages_column[:5])
print("Step slice demo from row 0 [session_duration, purchase_amount]:", step_slice_demo)
print("Users above purchase threshold:", high_value_users.shape[0])
print("Any high-value users?:", np.any(high_value_mask))
print("All users have positive session duration?:", np.all(positive_session_mask))


Sample indices: [  0  24 149 320]
Sample user rows:
 [[  1.  444.1   5.    0.    4. ]
 [ 25.  127.3   8.    0.    1. ]
 [150.  349.5   9.    0.    4. ]
 [321.  274.9   9.    0.    0. ]]
First user full row: [  1.  444.1   5.    0.    4. ]
Pages viewed column (first 5): [ 5.  3.  7. 15.  6.]
Step slice demo from row 0 [session_duration, purchase_amount]: [444.1   0. ]
Users above purchase threshold: 24
Any high-value users?: True
All users have positive session duration?: True


In [24]:
# This is what sklearn's StandardScaler does internally: center by the column mean and divide by the column std.
def normalize_columns(matrix):
    means = np.mean(matrix, axis=0)
    stds = np.std(matrix, axis=0)
    safe_stds = np.where(stds == 0, 1.0, stds)
    centered = np.subtract(matrix, means)
    normalized = np.divide(centered, safe_stds)
    return normalized, means, safe_stds


normalized_core, core_means, core_stds = normalize_columns(raw_feature_block)
pages_per_second = np.divide(
    raw_feature_block[:, [1]],
    np.maximum(raw_feature_block[:, [0]], 1.0),
)
purchase_sqrt = np.sqrt(raw_feature_block[:, [2]])
purchase_flag = np.multiply(raw_feature_block[:, [2]] > 0, 1.0).reshape(-1, 1)

feature_matrix = np.hstack(
    [
        normalized_core,
        pages_per_second,
        purchase_sqrt,
        purchase_flag,
    ]
)
feature_labels = [
    f"feature::{name}"
    for name in (
        "norm_session_duration",
        "norm_pages_viewed",
        "norm_purchase_amount",
        "pages_per_second",
        "sqrt_purchase_amount",
        "purchase_flag",
    )
]

feature_means = np.mean(feature_matrix, axis=0)
feature_stds = np.std(feature_matrix, axis=0)
feature_mins = np.min(feature_matrix, axis=0)
feature_maxs = np.max(feature_matrix, axis=0)
feature_diagnostics = np.vstack([feature_means, feature_stds, feature_mins, feature_maxs])
diagnostic_labels = ("mean", "std", "min", "max")

bias_row = np.array([[0.05, 0.05, 0.10, 0.01, 0.02, 0.03]], dtype=np.float64)
bias_matrix = np.tile(bias_row, (feature_matrix.shape[0], 1))
feature_matrix_with_bias = np.add(feature_matrix, bias_matrix)

print("Core feature means:", core_means)
print("Core feature stds:", core_stds)
print("Feature labels:", feature_labels)
print("Feature matrix shape:", feature_matrix.shape)
print("Feature matrix with tiled bias shape:", feature_matrix_with_bias.shape)
print("First engineered row (before bias):", feature_matrix[0])
print("First engineered row (after bias):", feature_matrix_with_bias[0])
print("All feature stds are non-zero after engineering?:", np.all(feature_stds > 0))
print("Feature diagnostics rows:", diagnostic_labels)
print(feature_diagnostics)


Core feature means: [493.622     7.692    15.14626]
Core feature stds: [793.10394632   6.85281957  44.44547151]
Feature labels: ['feature::norm_session_duration', 'feature::norm_pages_viewed', 'feature::norm_purchase_amount', 'feature::pages_per_second', 'feature::sqrt_purchase_amount', 'feature::purchase_flag']
Feature matrix shape: (500, 6)
Feature matrix with tiled bias shape: (500, 6)
First engineered row (before bias): [-0.06244074 -0.39283101 -0.34078297  0.01125873  0.          0.        ]
First engineered row (after bias): [-0.01244074 -0.34283101 -0.24078297  0.02125873  0.02        0.03      ]
All feature stds are non-zero after engineering?: True
Feature diagnostics rows: ('mean', 'std', 'min', 'max')
[[-1.60316205e-16 -3.73034936e-17  3.24185123e-17  3.45499229e-02
   1.62920988e+00  2.12000000e-01]
 [ 1.00000000e+00  1.00000000e+00  1.00000000e+00  3.77109768e-02
   3.53439318e+00  4.08724846e-01]
 [-5.84566502e-01 -9.76532351e-01 -3.40782975e-01  8.33333333e-03
   0.00000

### Module 3 (OLAP aggregation, matmul scoring, reverse-ETL output)


In [25]:
unique_regions = np.array(sorted(observed_region_ids))
region_report_rows = []

for region in unique_regions:
    region_mask = user_activity_struct["region_id"] == region
    region_numeric = user_activity_data[region_mask][:, 1:4]
    region_report_rows.append(
        np.array(
            [
                region,
                np.sum(region_numeric[:, 2], axis=0),
                np.mean(region_numeric[:, 0], axis=0),
                np.max(region_numeric[:, 1], axis=0),
                np.min(region_numeric[:, 0], axis=0),
                np.sum(region_mask),
            ],
            dtype=np.float64,
        )
    )

region_report = np.vstack(region_report_rows)
region_report_headers = (
    "region_id",
    "total_revenue",
    "avg_session_sec",
    "max_pages_viewed",
    "min_session_sec",
    "user_count",
)

weights_column = np.array([0.45, 0.25, 0.55, 0.15, 0.20, 0.10], dtype=np.float64).reshape(-1, 1)
weights_row = weights_column.T

engagement_scores = np.matmul(feature_matrix_with_bias, weights_column)
first_user_row_vector = feature_matrix_with_bias[0].reshape(1, -1)
first_user_matmul_score = np.matmul(first_user_row_vector, weights_column)
first_user_dot_score = np.dot(feature_matrix_with_bias[0], weights_row.ravel())

reverse_etl_output = np.hstack([user_activity_data, engagement_scores])
reverse_etl_headers = raw_column_labels + ["derived::engagement_score"]
top_user_indices = np.argsort(reverse_etl_output[:, -1])[-5:][::-1]

print("Region report headers:", region_report_headers)
print(region_report)
print("weights_row shape:", weights_row.shape)
print("weights_column shape:", weights_column.shape)
print("engagement_scores shape:", engagement_scores.shape)
print("First user matmul score:", first_user_matmul_score[0, 0])
print("First user dot-product score:", first_user_dot_score)
print("Reverse-ETL output headers:", reverse_etl_headers)
print("Top 5 users by engagement score:\n", reverse_etl_output[top_user_indices])


Region report headers: ('region_id', 'total_revenue', 'avg_session_sec', 'max_pages_viewed', 'min_session_sec', 'user_count')
[[0.00000000e+00 1.88931000e+03 5.03852980e+02 5.90000000e+01
  3.00000000e+01 1.51000000e+02]
 [1.00000000e+00 1.76808000e+03 4.13492913e+02 2.70000000e+01
  3.00000000e+01 1.27000000e+02]
 [2.00000000e+00 1.15086000e+03 4.09280000e+02 2.20000000e+01
  3.00000000e+01 9.50000000e+01]
 [3.00000000e+00 1.67468000e+03 5.86024691e+02 6.00000000e+01
  3.00000000e+01 8.10000000e+01]
 [4.00000000e+00 1.09020000e+03 6.92739130e+02 6.00000000e+01
  3.00000000e+01 4.60000000e+01]]
weights_row shape: (1, 6)
weights_column shape: (6, 1)
engagement_scores shape: (500, 1)
First user matmul score: -0.21354791324463446
First user dot-product score: -0.21354791324463446
Reverse-ETL output headers: ['raw::user_id', 'raw::session_duration_sec', 'raw::pages_viewed', 'raw::purchase_amount', 'raw::region_id', 'derived::engagement_score']
Top 5 users by engagement score:
 [[3.76000000

**Module 3 production note**

This score column is **derived data**, not a new system of record. In production, the OLAP-style aggregation would usually run in a warehouse such as Snowflake or BigQuery, and the reverse-ETL sync would push the final `engagement_score` back into product surfaces through a tool like Census or Hightouch so the application can rank, target, or personalize user experiences.


### Module 4 (Broadcasting experiments)


In [26]:
broadcast_base = feature_matrix_with_bias[:3, :3]

# Prediction 1: this works and stays shape (3, 3) because a length-3 row broadcasts across all rows.
experiment_1_row = np.array([10.0, 20.0, 30.0])
experiment_1 = broadcast_base + experiment_1_row

# Prediction 2: this works and stays shape (3, 3) because a (3, 1) column broadcasts across all columns.
experiment_2_col = np.array([[1.0], [2.0], [3.0]])
experiment_2 = broadcast_base * experiment_2_col

# Prediction 3: this fails because trailing dimensions 3 and 4 do not match, and neither side is 1.
experiment_3_bad = np.array([1.0, 2.0, 3.0, 4.0])
try:
    experiment_3 = broadcast_base + experiment_3_bad
    experiment_3_result = experiment_3
except ValueError as exc:
    experiment_3_result = str(exc)

print("Broadcast base shape:", broadcast_base.shape)
print("Experiment 1 shape:", experiment_1.shape)
print(experiment_1)
print("Experiment 2 shape:", experiment_2.shape)
print(experiment_2)
print("Experiment 3 result:", experiment_3_result)


Broadcast base shape: (3, 3)
Experiment 1 shape: (3, 3)
[[ 9.98755926 19.65716899 29.75921703]
 [ 9.68898553 19.36531832 29.75921703]
 [10.09876788 19.94901967 29.75921703]]
Experiment 2 shape: (3, 3)
[[-0.01244074 -0.34283101 -0.24078297]
 [-0.62202894 -1.26936336 -0.48156595]
 [ 0.29630365 -0.152941   -0.72234892]]
Experiment 3 result: operands could not be broadcast together with shapes (3,3) (4,) 


**Broadcasting reflection**

The first two experiments work because NumPy compares shapes from the trailing dimension backward and allows expansion when one side is `1`. A `(3,)` row can expand across three rows, and a `(3, 1)` column can expand across three columns. The third experiment fails because `(3, 3)` and `(4,)` cannot be aligned: the last dimensions are `3` and `4`, and neither is `1`.


### Module 5 (Systems concepts panel)

- **OLTP / system of record**: `user_events.csv` stands in for a PostgreSQL-style operational export. This is where low-latency writes and point reads happen for real users.
- **ETL flow**: Module 1 extracts the operational data, Module 2 transforms it into a feature matrix, and Module 3 loads derived outputs into an analytics-ready score table.
- **OLAP / warehouse thinking**: the per-region aggregation and engagement-score table are the kind of large-scan, aggregate-heavy outputs that belong in Snowflake or BigQuery, not on the operational database.
- **Derived data vs source data**: `user_activity_struct` remains the source-preserving raw export, while `feature_matrix_with_bias`, `region_report`, and `reverse_etl_output` are all derived artifacts.
- **Sushi principle**: I preserve the raw data alongside transformed features so downstream consumers can answer new questions later without losing optionality.
- **Data lake vs data warehouse**: a lake keeps raw or semi-structured data cheaply; a warehouse keeps cleaned, modeled tables optimized for analytical queries.
- **Role boundaries**: backend engineers own the application write path, data engineers own the ETL and score pipeline, and analytics engineers or data scientists consume the derived tables for reporting, experimentation, and modeling.


### Module 6 (Open questions panel)

The notebook is complete, but a few topics are still honestly shallow for me and deserve follow-up:

- **CAP theorem under a real network partition**: I can define the terms, but I still need a worked system example showing what choosing consistency over availability actually feels like in production.
- **SIMD and cache behavior from first principles**: I can name them in the performance chain, but I want to explain exactly how contiguous reads and vector instructions reduce CPU work at the hardware level.
- **Cloud vs self-hosting tradeoffs**: I understand the headline tradeoff between control and operational burden, but I want sharper instincts about when a team should switch from self-hosted infrastructure to managed cloud services.
- **`tokenizer?` and `canonical?` from my source notes**: these are still unresolved margin questions and should stay explicitly open until I can define them in context.
